# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,WEN,10,8,0.328475,2026-07-02 11:42:07+00:00
1,NVDA,6,4,0.114650,2026-07-01 23:56:45+00:00
2,NBIS,5,4,0.175800,2026-07-02 15:43:34+00:00
3,ZIM,15,1,0.993700,2026-07-02 19:59:29+00:00
4,RDDT,6,3,0.685667,2026-07-02 16:05:20+00:00
5,RKLB,4,3,0.091867,2026-07-02 14:34:07+00:00
6,MSFT,4,3,-0.382267,2026-07-02 23:22:59+00:00
7,MSTR,7,2,-0.335900,2026-06-30 08:48:46+00:00
8,CEG,7,1,0.945400,2026-06-28 16:14:31+00:00
9,QQQ,2,2,-0.170000,2026-07-02 21:25:20+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['WEN', 'NVDA', 'NBIS', 'ZIM']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
def safe_latest_value(in_df, column_name, caster=float):
    if in_df.empty or column_name not in in_df.columns:
        return None
    col_data = in_df[column_name].dropna()
    if col_data.empty:
        return None
    return caster(col_data.iloc[-1])


price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": safe_latest_value(features_df, "Close", float),
        "latest_rsi": safe_latest_value(features_df, "RSI", float),
        "latest_macd": safe_latest_value(features_df, "MACD", float),
        "return_21d": safe_latest_value(features_df, "21D Return", float),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,ZIM,251,25.57,51.397396,0.045731,0.01428,0.579632,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== WEN latest metrics =====
            Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                
2026-06-26   7.80  80470200            0.568064            7.240000             7.288630  58.496767  0.001739   0.147059    0.041389
2026-06-29   8.26  36706200            0.576582            7.245000             7.310889  62.599850  0.095195   0.338736    0.101333
2026-06-30   8.29  30055800            0.591022            7.254000             7.333323  62.857744  0.169724   0.324281    0.076623
2026-07-01   8.94  30143600            0.658860            7.290667             7.370134  68.005554  0.278033   0.137405    0.138853
2026-07-02   8.60  30216363            0.685004            7.306667             7.398309  63.080628  0.332599   0.173261    0.192788

===== NVDA latest metrics =====
    

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== WEN regression results =====
            Close  Predicted Close
Date                              
2026-06-18   6.80         6.846850
2026-06-22   6.17         6.369728
2026-06-23   6.26         6.399432
2026-06-24   7.86         7.258259
2026-06-25   7.33         7.186384
2026-06-26   7.80         7.570384
2026-06-29   8.26         8.157589
2026-06-30   8.29         8.233999
2026-07-01   8.94         8.381362
2026-07-02   8.60         8.461968

===== NVDA regression results =====
                 Close  Predicted Close
Date                                   
2026-06-18  210.690002       211.798329
2026-06-22  208.649994       210.464666
2026-06-23  200.039993       204.366557
2026-06-24  199.000000       203.232785
2026-06-25  195.740005       200.326419
2026-06-26  192.529999       196.805261
2026-06-29  194.970001       197.399520
2026-06-30  200.089996       201.403054
2026-07-01  197.580002       198.580882
2026-07-02  194.830002       196.393046

===== NBIS regression resu

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "WEN",
    "NVDA",
    "NBIS",
    "ZIM"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
